In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

SOURCE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.products"
TEST_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.test_products_schema"

##### All experiments run on a deep clone, so the real silver table is never modified

In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {TEST_TABLE} DEEP CLONE {SOURCE_TABLE}")

##### Schema enforcement. A write with a column that does not exist in the table is rejected

In [0]:
extra_column_df = spark.table(TEST_TABLE).limit(1).withColumn("region", F.lit("EU"))

try:
    extra_column_df.write.mode("append").saveAsTable(TEST_TABLE)
except Exception as error:
    rejection_message = str(error).splitlines()[0]

rejection_message

##### Schema evolution: adding a column 
##### The same kind of write succeeds when `mergeSchema` explicitly allows the new column

In [0]:
new_column_df = spark.table(TEST_TABLE).limit(0).withColumn("stock_qty", F.lit(None).cast("int"))

new_column_df.write.mode("append").option("mergeSchema", "true").saveAsTable(TEST_TABLE)


![image_1791296220796.png](./screenshots/image_1791296220796.png)

##### Type widening `INT` is widened to `BIGINT` without rewriting data

In [0]:
spark.sql(f"ALTER TABLE {TEST_TABLE} SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')")
spark.sql(f"ALTER TABLE {TEST_TABLE} ALTER COLUMN stock_qty TYPE BIGINT")


![image_1791296246627.png](./screenshots/image_1791296246627.png "image_1791296246627.png")

##### Column mapping: rename

In [0]:
spark.sql(f"""
ALTER TABLE {TEST_TABLE} SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5'
)
""")

spark.sql(f"ALTER TABLE {TEST_TABLE} RENAME COLUMN stock_qty TO stock_quantity")

![image_1791296278564.png](./screenshots/image_1791296278564.png "image_1791296278564.png")

##### Column mapping: drop. Dropping a column is a metadata-only change.

In [0]:
spark.sql(f"ALTER TABLE {TEST_TABLE} DROP COLUMN stock_quantity")

![image_1791296289274.png](./screenshots/image_1791296289274.png "image_1791296289274.png")

##### Cleanup

In [0]:
spark.sql(f"DROP TABLE {TEST_TABLE}")

## Data contracts vs silent schema evolution

**Silent evolution** (`mergeSchema` / `autoMerge` always on) lets the producer change the table schema without anyone agreeing to it - consumers find out only when something breaks.

**A data contract** is an explicit agreement between producer and consumer on columns, types, required fields and quality rules. A new optional column is a non-breaking change; a removed, renamed or narrowed column is breaking. Schema changes happen only after the contract is updated and reviewed.

**In this lab:** Bronze accepts the source as is, while Silver is the contract boundary - pipelines select an explicit list of columns. A new source column reaches Bronze automatically, but reaches Silver only after it is added to the pipeline's column list via a reviewed commit; the MERGE then adds it with `withSchemaEvolution()`.